In [1]:
import os
import time
import yaml
import numpy as np
import torch
from torch.utils.data import TensorDataset
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt


import pandas as pd
import gendata
from models.wdgrl import WDGRL

c:\Users\Asus\anaconda3\envs\dann\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
source = pd.read_csv("obesitylevel/gender0.csv")
target = pd.read_csv("obesitylevel/gender1.csv")

In [3]:
y_source = source['NObeyesdad']
y_target = target['NObeyesdad']
X_source = source.drop(columns=['NObeyesdad'])
X_target = target.drop(columns=['NObeyesdad'])

In [4]:
print(y_source.value_counts())   
print(y_target.value_counts())

NObeyesdad
4    323
0    173
2    156
5    145
1    141
6    103
3      2
Name: count, dtype: int64
NObeyesdad
3    295
2    195
6    187
1    146
5    145
0     99
4      1
Name: count, dtype: int64


In [ ]:
# print(X_source.shape)
# print(X_target.shape)

(1043, 19)
(1068, 19)


In [ ]:
# # train/test split (60% train, 40% test) for source and target without extra imports
# train_frac = 0.6
# random_state = 42

# # source split
# _source_train_idx = X_source.sample(frac=train_frac, random_state=random_state).index
# source_train = X_source.loc[_source_train_idx].reset_index(drop=True)
# source_test = X_source.drop(_source_train_idx).reset_index(drop=True)

# # target split
# _target_train_idx = X_target.sample(frac=train_frac, random_state=random_state).index
# target_train = X_target.loc[_target_train_idx].reset_index(drop=True)
# target_test = X_target.drop(_target_train_idx).reset_index(drop=True)

# # quick sanity check
# print("source:", X_source.shape, "->", "train:", source_train.shape, "test:", source_test.shape)
# print("target:", X_target.shape, "->", "train:", target_train.shape, "test:", target_test.shape)

source: (1043, 19) -> train: (626, 19) test: (417, 19)
target: (1068, 19) -> train: (641, 19) test: (427, 19)


In [ ]:
# source_train.to_csv("obesitylevel/gender0_train.csv", index=False)
# source_test.to_csv("obesitylevel/gender0_test.csv", index=False)
# target_train.to_csv("obesitylevel/gender1_train.csv", index=False)
# target_test.to_csv("obesitylevel/gender1_test.csv", index=False)

In [22]:
X_source = pd.read_csv("obesitylevel/gender0_train.csv")
X_target = pd.read_csv("obesitylevel/gender1_train.csv")
# to numpy
X_source = X_source.to_numpy(dtype=np.float64)
X_target = X_target.to_numpy(dtype=np.float64)

In [23]:
ns, nt = X_source.shape[0], X_target.shape[0]
d = X_source.shape[1]

In [24]:
xs = torch.from_numpy(X_source).double()
xt = torch.from_numpy(X_target).double()

source_dataset = TensorDataset(xs)
target_dataset = TensorDataset(xt)

print(d)

19


In [25]:
# ==== WDGRL model ====
final_model = WDGRL(
    input_dim=d,
    encoder_hidden_dims=[500,100],
    critic_hidden_dims=[100],
    alpha1=0.0001,
    alpha2=0.0001,
    seed=42,
)

In [26]:
# ==== Logging setup ====
timestamp = time.strftime("%Y%m%d-%H%M%S")
log_dir = os.path.join("logs", timestamp)
os.makedirs(log_dir, exist_ok=True)

log_file = os.path.join(log_dir, "results.txt")
log_loss = final_model.train(
    source_dataset,
    target_dataset,
    num_epochs=6000,
    gamma=10,
    dc_iter=5,
    batch_size=32,
    # early_stopping=True,
    model_path=log_dir,
)

Epoch:   0%|          | 0/6000 [00:00<?, ?it/s]c:\Users\Asus\anaconda3\envs\dann\Lib\site-packages\torch\autograd\graph.py:829: UserWarning: Attempting to run cuBLAS, but there was no current CUDA context! Attempting to set the primary context... (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\cuda\CublasHandlePool.cpp:179.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass
Epoch: 100%|██████████| 6000/6000 [01:50<00:00, 54.54it/s]


In [27]:
final_model.save_model(log_dir)

# ==== Save logs ====
total_loss = log_loss["loss"]

Encoder and Critic saved to logs\20251018-143239


In [28]:
epochs = range(1, len(total_loss) + 1)

plt.figure(figsize=(14, 6))
plt.plot(epochs, total_loss, linestyle='-', color='blue')
plt.title("Loss over Epochs")
plt.xlabel("Epoch")
plt.ylabel("Loss")
# plt.ylim(0, 1.0)
plt.grid(True)
plt.savefig(os.path.join(log_dir, "loss.png"))
plt.close()